# Testing LV1 — `pfm prep LV1 :: build_atm`

Demonstrates how to produce `LV1_ATM_FORCING.nc` via the v3 pipeline.

`build_atm` is a relocation of v2's three atm subprocess hops:
1. `get_atm_data_as_dict` — fetch ECMWF grib2 from CDIP, decode → one ATM dict (pickle in `lv1_forc_dir`)
2. `get_atm_data_on_roms_grid` — regrid to LV1, rotate winds to xi/eta
3. `atm_roms_dict_to_netcdf` — write `LV1_ATM_FORCING.nc`

Run this notebook with the **Python (pfm_v3-env)** kernel.

⚠️ The download step in build_atm takes 5–15 minutes on a fresh cache.

## 1. Imports + paths

In [1]:
from pathlib import Path
from datetime import datetime

from pfm.config import Config, OP_ROOT, V3_ROOT
from pfm.lv1    import build_atm

OP_PICKLE = OP_ROOT / 'forecast_info.pkl'              # operational config (read-only)
V3_PICKLE = V3_ROOT / 'forecast_info_v3.pkl'           # v3-redirected config
print('operational pickle:', OP_PICKLE, '  exists:', OP_PICKLE.exists())
print('v3 root          :', V3_ROOT)

operational pickle: /scratch/PFM_Simulations/forecast_info.pkl   exists: True
v3 root          : /scratch/PFM_v3_Simulations


## 2. Load the operational config, redirect outputs under V3_ROOT, persist

In [2]:
cfg_op = Config.from_pickle(OP_PICKLE)
print(cfg_op)                                          # <Config run_type='forecast' keys=...>
print('cycle start :', cfg_op.sim_start_time)
print('cycle end   :', cfg_op.sim_end_time)

cfg_v3 = cfg_op.redirect_outputs_to()                  # every output path -> V3_ROOT/...
#cfg_v3._sim_start_time = '2026-06-24 12:00:00'
#cfg_v3._sim_end_time = '2026-06-29 12:00:00'
n_redirected = len(cfg_v3.output_path_keys())
print(f'redirected {n_redirected} output paths under {V3_ROOT}')

# persist; build_atm needs an on-disk pickle because the legacy code
# (sdpm_py_util/atm_functions.py) reads paths from a pickle
V3_PICKLE.parent.mkdir(parents=True, exist_ok=True)
cfg_v3.to_pickle(V3_PICKLE)
print(f'wrote v3 config: {V3_PICKLE}')

# confirm LV1 outputs will land under v3 root, not operational
print(f'\nLV1 atm out  : {cfg_v3.forc_dir(1) / cfg_v3.raw["lv1_atm_file"]}')
print(f'LV1 grid (input, unchanged): {cfg_v3.grid_file(1)}')


<Config run_type='forecast' keys=184>
cycle start : 2026-06-24 00:00:00
cycle end   : 2026-06-29 00:00:00
redirected 40 output paths under /scratch/PFM_v3_Simulations
wrote v3 config: /scratch/PFM_v3_Simulations/forecast_info_v3.pkl

LV1 atm out  : /scratch/PFM_v3_Simulations/LV1_Forecast/Forc/LV1_ATM_FORCING.nc
LV1 grid (input, unchanged): /scratch/PFM_Simulations/Grids/GRID_SDTJRE_LV1_rx020_hmask.nc


## 3. Run `build_atm`

This invokes the three legacy subprocess hops in order, streaming their
stdout in real time so you can watch the ECMWF download progress.

In [3]:
t0 = datetime.now()
atm_nc = build_atm(cfg_v3, V3_PICKLE)
print(f'\n=== build_atm wrote: {atm_nc}')
print(f'=== total elapsed  : {datetime.now() - t0}')
print(f'=== size           : {atm_nc.stat().st_size / (1<<20):.1f} MB')


[build_atm] step 1/3 :: get_atm_data_as_dict
[legacy] /home/mspydell/anaconda3/envs/pfm_v3-env/bin/python -u -W ignore atm_functions.py get_atm_data_as_dict /scratch/PFM_v3_Simulations/forecast_info_v3.pkl  (cwd=/home/mspydell/models/PFM_root/PFM/sdpm_py_util)
trying the new more robust method of getting and using ecmwf data from cdip...
getting the ecmwf data from cdip for the 2026062400 forecast...
getting data directly from ecmwf!

🚀 Starting Download Batch Attempt #1 (101 files remaining)...
⚠️ Ghost file detected (0 bytes): ecmwf_dataT1D0624002026062410001. Cleaning up for retry.
⚠️ Ghost file detected (0 bytes): ecmwf_dataT1D0624002026062411001. Cleaning up for retry.
⚠️ Ghost file detected (0 bytes): ecmwf_dataT1D0624002026062412001. Cleaning up for retry.
⚠️ Ghost file detected (0 bytes): ecmwf_dataT1D0624002026062413001. Cleaning up for retry.
⚠️ Ghost file detected (0 bytes): ecmwf_dataT1D0624002026062414001. Cleaning up for retry.
⚠️ Ghost file detected (0 bytes): ecmwf_dat

Ignoring index file '/scratch/PFM_v3_Simulations/ecmwf_dataT1D0624002026062400011.5b7b6.idx' older than GRIB file
Ignoring index file '/scratch/PFM_v3_Simulations/ecmwf_dataT1D0624002026062400011.5b7b6.idx' older than GRIB file
Ignoring index file '/scratch/PFM_v3_Simulations/ecmwf_dataT1D0624002026062401001.5b7b6.idx' older than GRIB file
Ignoring index file '/scratch/PFM_v3_Simulations/ecmwf_dataT1D0624002026062402001.5b7b6.idx' older than GRIB file
Ignoring index file '/scratch/PFM_v3_Simulations/ecmwf_dataT1D0624002026062403001.5b7b6.idx' older than GRIB file
Ignoring index file '/scratch/PFM_v3_Simulations/ecmwf_dataT1D0624002026062404001.5b7b6.idx' older than GRIB file
Ignoring index file '/scratch/PFM_v3_Simulations/ecmwf_dataT1D0624002026062405001.5b7b6.idx' older than GRIB file
Ignoring index file '/scratch/PFM_v3_Simulations/ecmwf_dataT1D0624002026062406001.5b7b6.idx' older than GRIB file
Ignoring index file '/scratch/PFM_v3_Simulations/ecmwf_dataT1D0624002026062407001.5b7b6.


ecmwf 1st ATM dict saved with pickle.
return code: 0 (0=good)
...done.
deleting grb and idx files from  /scratch/PFM_v3_Simulations/ecmwf_data  ...
...done

going from ecmwf variables to roms variables...

ecmwf ATM dict roms vars saved with pickle.
return code: 0 (0=good)
...done.
  took 0:05:44.185396

[build_atm] step 2/3 :: get_atm_data_on_roms_grid (LV1)
[legacy] /home/mspydell/anaconda3/envs/pfm_v3-env/bin/python -u -W ignore atm_functions.py get_atm_data_on_roms_grid 1 /scratch/PFM_v3_Simulations/forecast_info_v3.pkl  (cwd=/home/mspydell/models/PFM_root/PFM/sdpm_py_util)
in the atm function!
the atm pickle file is:
/scratch/PFM_v3_Simulations/LV1_Forecast/Forc/atm_tmp_pckl_file.pkl
and this file exists!
loading /scratch/PFM_v3_Simulations/LV1_Forecast/Forc/atm_tmp_pckl_file.pkl ...
interpolating to LV1 grid...
... done.
rotating velocities to LV 1 roms directions...
...done.
saving atm to LV1 pkl file...

ATM on roms grid dict saved with pickle.
  took 0:00:03.978331

[build_at

## 4. Inspect the output

In [4]:
import netCDF4 as nc
with nc.Dataset(atm_nc) as d:
    print('dims    :', dict((k, len(v)) for k, v in d.dimensions.items()))
    print('vars    :', [n for n in d.variables if n not in d.dimensions])
    if 'wind_time' in d.variables:
        t = d.variables['wind_time']
        print(f'wind_time: n={len(t)}  units={getattr(t, "units", "?")!r}  '
              f'range=[{float(t[0]):.4f}, {float(t[-1]):.4f}]')
    for v in ('Uwind', 'Vwind', 'Tair', 'swrad'):
        if v in d.variables:
            arr = d.variables[v]
            print(f'{v:8s}  shape={arr.shape}  units={getattr(arr, "units", "?")}')

dims    : {'tair_time': 101, 'er': 390, 'xr': 253, 'pair_time': 101, 'qair_time': 101, 'wind_time': 101, 'rain_time': 102, 'srf_time': 102, 'lrf_time': 102, 'time': 101}
vars    : ['Tair', 'Pair', 'Qair', 'Uwind', 'Vwind', 'rain', 'swrad', 'lwrad', 'lwrad_down', 'lat', 'lon', 'ocean_time']
wind_time: n=101  units='days'  range=[10036.0000, 10041.0000]
Uwind     shape=(101, 390, 253)  units=m/s
Vwind     shape=(101, 390, 253)  units=m/s
Tair      shape=(101, 390, 253)  units=degrees C
swrad     shape=(102, 390, 253)  units=W/m^2


## 5. CLI equivalent

Everything above is also doable from the shell — useful for cron / sbatch
wrappers.  Cells 2 + 3 are equivalent to:

```bash
# 1.  build the v3-redirected pickle from the operational state
python tools/pfm_v3_from_operational.py 1   # writes V3_PICKLE, validates redirect

# 2.  run just the atm-build step
python -m pfm prep LV1 --config /scratch/PFM_v3_Simulations/forecast_info_v3.pkl \
                       --step build_atm
```

Or as a one-shot full LV1 prep (when the rest of the steps land):

```bash
python -m pfm prep LV1 --config /scratch/PFM_v3_Simulations/forecast_info_v3.pkl
```

## 6. Compare v3 vs operational (optional)

Once the v3 atm.nc is written, you can diff it against the operational
one to confirm they agree:

In [5]:
import numpy as np

op_atm = cfg_op.forc_dir(1) / cfg_op.raw['lv1_atm_file']
v3_atm = cfg_v3.forc_dir(1) / cfg_v3.raw['lv1_atm_file']
print(f'operational : {op_atm}  exists={op_atm.exists()}')
print(f'v3          : {v3_atm}  exists={v3_atm.exists()}')

if op_atm.exists() and v3_atm.exists():
    with nc.Dataset(op_atm) as a, nc.Dataset(v3_atm) as b:
        for v in ('Uwind', 'Vwind', 'Tair', 'swrad', 'Pair', 'Qair'):
            if v in a.variables and v in b.variables:
                d = np.asarray(a.variables[v][:]) - np.asarray(b.variables[v][:])
                print(f'  {v:8s}  max|op-v3|={float(np.max(np.abs(d))):.4g}  '
                      f'rmse={float(np.sqrt(np.mean(d**2))):.4g}')

operational : /scratch/PFM_Simulations/LV1_Forecast/Forc/LV1_ATM_FORCING.nc  exists=True
v3          : /scratch/PFM_v3_Simulations/LV1_Forecast/Forc/LV1_ATM_FORCING.nc  exists=True
  Uwind     max|op-v3|=0  rmse=0
  Vwind     max|op-v3|=0  rmse=0
  Tair      max|op-v3|=0  rmse=0
  swrad     max|op-v3|=0  rmse=0
  Pair      max|op-v3|=0  rmse=0
  Qair      max|op-v3|=0  rmse=0
